In [ ]:
%config InlineBackend.figure_formats = ['retina']

import sys
sys.path.insert(0, "../src")

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm.auto import tqdm

from generate_cases import *
from seg_metrics import compute_segmentation_metrics

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.figsize": (6.5, 4.5),   # letter-paper single column width
    "figure.dpi": 150,
    "axes.prop_cycle": plt.cycler(color=["black", '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']),
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## Part 1: Segmentation methods vs. ground truth (label_wg), full ProstateX cohort

In [ ]:
# Collect all the data metadata
csv_path = Path("../data/raw/github/prostatex_masks/index.csv")
prostatex_cases = pd.read_csv(csv_path)
assert not prostatex_cases["ID"].duplicated().any(), "Duplicate ProstateX IDs in index.csv!"

# Convert all mask/image paths to absolute paths
for col in ["t2", "label_wg", "label_tz", "label_pz"]:
    prostatex_cases[col] = prostatex_cases[col].apply(lambda x: str((csv_path.parent / x).resolve()))

# Concatenate locally run predictions

# monai bundle output mirrors the source layout: Files/prostate/Images/<t2 stem>/<t2 stem>_trans.nii.gz
prostatex_cases["Prostate158_adams22"] = prostatex_cases["t2"].apply(
    lambda p: f"../results/prostatex_pred_monai_prostate/Files/prostate/Images/{Path(p).name.removesuffix('.nii.gz')}/{Path(p).name.removesuffix('.nii.gz')}_trans.nii.gz"
)
assert all(Path(p).exists() for p in prostatex_cases["Prostate158_adams22"]), "Some predictions are missing!"

prostatex_cases["Prostate158_nnunet"] = prostatex_cases["ID"].apply(lambda id: f"../results/prostatex_pred_prostate158_3d_fullres/{id}.nii.gz")
assert all(Path(p).exists() for p in prostatex_cases["Prostate158_nnunet"]), "Some predictions are missing!"

prostatex_cases["pooled_nnunet"] = prostatex_cases["ID"].apply(lambda id: f"../results/prostatex_pred_pooled2_3d_fullres/{id}.nii.gz")
assert all(Path(p).exists() for p in prostatex_cases["pooled_nnunet"]), "Some predictions are missing!"

prostatex_cases


In [ ]:
from concurrent.futures import ProcessPoolExecutor
import os

def _row_volumes(row):
    import nibabel as nib
    import numpy as np
    for col, path in row.items():
        if col in ("ID", "t2", "label_tz", "label_pz"):
            continue
        img = nib.load(path)
        data = np.asarray(img.dataobj)
        voxel_vol_mm3 = np.prod(img.header.get_zooms()[:3])
        row[col] = (data > 0).sum() * voxel_vol_mm3 / 1000.0
    return row

rows = prostatex_cases.to_dict("records")

with ProcessPoolExecutor(os.cpu_count() - 1) as pool:
    results = list(tqdm(pool.map(_row_volumes, rows), total=len(rows)))

volumes = pd.DataFrame(results)
volumes = volumes.drop(columns=["label_tz", "label_pz"])
volumes["gt"] = volumes["label_wg"]

volumes


In [ ]:
from concurrent.futures import ProcessPoolExecutor
import os
from itertools import combinations

SEG_COLS = [
    "label_wg",
    "Prostate158_adams22",
    "Prostate158_nnunet",
    "pooled_nnunet",
]

def _pairwise_dsc_case(row):
    import SimpleITK as sitk
    import numpy as np

    ref_img = sitk.ReadImage(str(row[SEG_COLS[0]]), sitk.sitkUInt8)
    masks = {SEG_COLS[0]: sitk.GetArrayFromImage(ref_img).ravel() > 0}
    for col in SEG_COLS[1:]:
        img = sitk.ReadImage(str(row[col]), sitk.sitkUInt8)
        needs_resample = (
            img.GetSize() != ref_img.GetSize()
            or not np.allclose(img.GetSpacing(), ref_img.GetSpacing(), rtol=1e-4)
            or not np.allclose(img.GetOrigin(), ref_img.GetOrigin(), rtol=1e-4)
        )
        if needs_resample:
            img = sitk.Resample(img, ref_img, sitk.Transform(),
                                sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
        masks[col] = sitk.GetArrayFromImage(img).ravel() > 0

    results = {"ID": row["ID"]}
    for m1, m2 in combinations(SEG_COLS, 2):
        a, b = masks[m1], masks[m2]
        denom = int(a.sum()) + int(b.sum())
        results[f"{m1}||{m2}"] = (
            2 * int((a & b).sum()) / denom if denom > 0 else float("nan")
        )
    return results

rows = prostatex_cases.to_dict("records")
with ProcessPoolExecutor(max_workers=os.cpu_count() - 1) as pool:
    dsc_records = list(tqdm(pool.map(_pairwise_dsc_case, rows), total=len(rows)))

dsc_df = pd.DataFrame(dsc_records)

dsc_matrix = pd.DataFrame(np.ones((len(SEG_COLS), len(SEG_COLS))),
                           index=SEG_COLS, columns=SEG_COLS)
for m1, m2 in combinations(SEG_COLS, 2):
    val = dsc_df[f"{m1}||{m2}"].mean()
    dsc_matrix.loc[m1, m2] = val
    dsc_matrix.loc[m2, m1] = val

print(f"Mean pairwise DSC across {len(prostatex_cases)} cases\n")
dsc_matrix.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format(precision=3)


In [ ]:
methods = [
    "Prostate158_adams22",
    "Prostate158_nnunet",
    "pooled_nnunet",
]

ncols = 3
nrows = -(-len(methods) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(8, 3), sharex=True, sharey=True)
axes = axes.flatten()

for ax, method in zip(axes, methods):
    diff = volumes[method] - volumes["gt"]
    ax.scatter(volumes["gt"], diff, s=5, alpha=0.5)
    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_title(method, fontsize=9)
    ax.set_xlabel("Ground truth volume (mL)")
    ax.set_ylabel("Difference from GT (mL)")

for ax in fig.axes:
    ax.tick_params(labelleft=True, labelbottom=True)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

for ax in axes[len(methods):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
rows = {}
for method in methods:
    diff = volumes[method] - volumes["gt"]
    m, s = diff.mean(), diff.std()
    desc = diff.describe()
    rows[method] = {
        "mean": m,
        "95% LoA": f"[{m - 1.96*s:.1f}, {m + 1.96*s:.1f}]",
        "min": desc["min"], "25%": desc["25%"], "50%": desc["50%"],
        "75%": desc["75%"], "max": desc["max"], "N": int(desc["count"])
    }

df = pd.DataFrame(rows).T

numeric_cols = ["mean", "min", "25%", "50%", "75%", "max"]
df[numeric_cols] = df[numeric_cols].astype(float)

fmt = {c: "{:.1f}" for c in numeric_cols}
fmt["N"] = "{:.0f}"

df.style.format(fmt)

In [ ]:
import pingouin as pg
from itertools import combinations

icc_methods = ["gt"] + methods
n = len(icc_methods)
icc_vals = pd.DataFrame(np.ones((n, n)), index=icc_methods, columns=icc_methods)

for m1, m2 in combinations(icc_methods, 2):
    long = pd.concat([
        volumes[["ID", m1]].rename(columns={m1: "volume"}).assign(rater=m1),
        volumes[["ID", m2]].rename(columns={m2: "volume"}).assign(rater=m2),
    ]).dropna(subset=["volume"])

    res = pg.intraclass_corr(data=long, targets="ID", raters="rater", ratings="volume", nan_policy='omit')
    row = res.set_index("Type").loc["ICC(A,1)"]   # two-way mixed, absolute agreement
    icc = row["ICC"]

    icc_vals.loc[m1, m2] = icc
    icc_vals.loc[m2, m1] = icc

print("ICC(A,1) — two-way mixed, absolute agreement (volume, mL)\n")
icc_vals.style.background_gradient(cmap="RdYlGn", vmin=0.8, vmax=1).format(precision=3)


## Part 2: Merge PICAI clinical metadata via ProstateX-mapping.json

Some ProstateX cases have a PICAI clinical-data counterpart (patient_id/study_id, with prostate_volume, PSA, etc.),
linked via `ProstateX-mapping.json`. The mapping key embeds a scan date the ProstateX `index.csv` ID does not carry,
so some ProstateX IDs map to more than one PICAI target (e.g. `ProstateX-0025` has two mapping entries under
different dates). We treat those as ambiguous and drop them rather than guessing which scan date is correct.
Cases without any PICAI counterpart are kept with blank (NaN) clinical columns.

In [ ]:
import json

mapping = json.load(open("../data/raw/zenodo/picai/picai_labels/additional_resources/ProstateX-mapping.json"))

# key format "ProstateX-XXXX_MM-DD-YYYY" -> "patient_id_study_id"; strip date to get the ProstateX ID
map_df = pd.DataFrame(
    [(k.split("_")[0], v) for k, v in mapping.items()],
    columns=["ID", "picai_key"],
)

# Explicit duplicate check + report
dupes = map_df[map_df["ID"].duplicated(keep=False)]
print(f"{dupes['ID'].nunique()} ProstateX IDs have ambiguous PICAI mappings — dropping (left blank):")
display(dupes.sort_values("ID"))
map_df = map_df[~map_df["ID"].isin(dupes["ID"])]

map_df[["patient_id", "study_id"]] = map_df["picai_key"].str.split("_", expand=True).astype(int)

marksheet = pd.read_csv("../data/raw/zenodo/picai/picai_labels/clinical_information/marksheet.csv")

volumes = volumes.merge(map_df[["ID", "patient_id", "study_id"]], on="ID", how="left")
volumes = volumes.merge(marksheet, on=["patient_id", "study_id"], how="left")

assert not volumes["ID"].duplicated().any(), "Merge produced duplicate ProstateX IDs!"
volumes


In [ ]:
clinical = volumes.dropna(subset=["psa", "prostate_volume"]).copy()
print(f"{len(clinical)} / {len(volumes)} ProstateX cases have matched PICAI clinical data.")

methods_with_manual = ["prostate_volume", "Prostate158_adams22", "Prostate158_nnunet", "pooled_nnunet"]
clinical


### Volume comparison, matched subset (adds manual/ellipsoid measurement)

In [ ]:
ncols = 4
nrows = -(-len(methods_with_manual) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(8, 3), sharex=True, sharey=True)
axes = axes.flatten()

# cases with >xxx mL error in ANY method get labeled on every subplot
outlier_mask = pd.Series(False, index=clinical.index)
for method in methods_with_manual:
    outlier_mask |= (clinical[method] - clinical["gt"]).abs() > 35

for ax, method in zip(axes, methods_with_manual):
    diff = clinical[method] - clinical["gt"]
    ax.scatter(clinical["gt"], diff, s=5, alpha=0.5)
    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_title({"prostate_volume": "Manual"}.get(method, method), fontsize=9)
    ax.set_xlabel("Ground truth volume (mL)")

    for idx in clinical.index[outlier_mask]:
        ax.annotate(
            clinical.loc[idx, "ID"][-3:],
            (clinical.loc[idx, "gt"], diff.loc[idx]),
            fontsize=6, color="red",
            xytext=(3, 3), textcoords="offset points",
        )

axes[0].set_ylabel("Difference from GT (mL)")

for ax in fig.axes:
    ax.tick_params(labelleft=True, labelbottom=True)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

for ax in axes[len(methods_with_manual):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
rows = {}
for method in methods_with_manual:
    diff = clinical[method] - clinical["gt"]
    m, s = diff.mean(), diff.std()
    desc = diff.describe()
    rows[{"prostate_volume": "Manual"}.get(method, method)] = {
        "mean": m,
        "95% LoA": f"[{m - 1.96*s:.1f}, {m + 1.96*s:.1f}]",
        "min": desc["min"], "25%": desc["25%"], "50%": desc["50%"],
        "75%": desc["75%"], "max": desc["max"], "N": int(desc["count"])
    }

df = pd.DataFrame(rows).T

numeric_cols = ["mean", "min", "25%", "50%", "75%", "max"]
df[numeric_cols] = df[numeric_cols].astype(float)

fmt = {c: "{:.1f}" for c in numeric_cols}
fmt["N"] = "{:.0f}"

df.style.format(fmt)

In [ ]:
icc_methods_clinical = ["gt"] + methods_with_manual
n = len(icc_methods_clinical)
icc_vals_clinical = pd.DataFrame(np.ones((n, n)), index=icc_methods_clinical, columns=icc_methods_clinical)

for m1, m2 in combinations(icc_methods_clinical, 2):
    long = pd.concat([
        clinical[["ID", m1]].rename(columns={m1: "volume"}).assign(rater=m1),
        clinical[["ID", m2]].rename(columns={m2: "volume"}).assign(rater=m2),
    ]).dropna(subset=["volume"])

    res = pg.intraclass_corr(data=long, targets="ID", raters="rater", ratings="volume", nan_policy='omit')
    row = res.set_index("Type").loc["ICC(A,1)"]
    icc = row["ICC"]

    icc_vals_clinical.loc[m1, m2] = icc
    icc_vals_clinical.loc[m2, m1] = icc

print("ICC(A,1) — two-way mixed, absolute agreement (volume, mL, matched subset)\n")
icc_vals_clinical.style.background_gradient(cmap="RdYlGn", vmin=0.8, vmax=1).format(precision=3)


### PSAD comparison, matched subset, restricted to clinically relevant range (GT PSAD < 0.3)

In [ ]:
psad = clinical.copy()
for method in methods_with_manual:
    psad[method] = psad["psa"] / clinical[method]
psad["gt"] = psad["psa"] / clinical["gt"]

psad = psad[psad["gt"] < 0.3]  # restrict to clinically relevant range
print(f"{len(psad)} / {len(clinical)} matched cases have GT PSAD < 0.3.")
psad


In [ ]:
import matplotlib.ticker as ticker

ncols = 4
nrows = -(-len(methods_with_manual) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(8, 2), sharex=True, sharey=True)
axes = axes.flatten()

# cases with >0.1 ng/mL² error in ANY method get labeled on every subplot
outlier_mask = pd.Series(False, index=psad.index)
for method in methods_with_manual:
    outlier_mask |= (psad[method] - psad["gt"]).abs() > 0.09

for ax, method in zip(axes, methods_with_manual):
    diff = psad[method] - psad["gt"]
    ax.scatter(psad["gt"], diff, s=5, alpha=0.5)
    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_title({"prostate_volume": "Manual"}.get(method, method), fontsize=9)
    ax.set_xlabel("GT PSAD (ng/mL²)")

    for idx in psad.index[outlier_mask]:
        ax.annotate(
            psad.loc[idx, "ID"][-3:],
            (psad.loc[idx, "gt"], diff.loc[idx]),
            fontsize=6, color="red",
            xytext=(3, 3), textcoords="offset points",
        )

axes[0].set_ylabel("Difference from GT PSAD\n(ng/mL²)")

for ax in fig.axes:
    ax.tick_params(labelleft=True, labelbottom=True)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    formatter = ticker.ScalarFormatter()
    formatter.set_powerlimits((-np.inf, np.inf))
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xlim([0.0, 0.3])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.1))
    # ax.set_ylim([-0.3, 0.3])
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    # ax.axvline(0.15, color="r", linewidth=0.8, linestyle="--")

for ax in axes[len(methods_with_manual):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
rows = {}
for method in methods_with_manual:
    diff = psad[method] - psad["gt"]
    m, s = diff.mean(), diff.std()
    desc = diff.describe()
    rows[{"prostate_volume": "Manual"}.get(method, method)] = {
        "mean": m,
        "95% LoA": f"[{m - 1.96*s:.1f}, {m + 1.96*s:.1f}]",
        "min": desc["min"], "25%": desc["25%"], "50%": desc["50%"],
        "75%": desc["75%"], "max": desc["max"], "N": int(desc["count"])
    }

df = pd.DataFrame(rows).T

numeric_cols = ["mean", "min", "25%", "50%", "75%", "max"]
df[numeric_cols] = df[numeric_cols].astype(float)

fmt = {c: "{:.3f}" for c in numeric_cols}
fmt["N"] = "{:.0f}"

df.style.format(fmt)

In [ ]:
icc_methods_psad = ["gt"] + methods_with_manual
n = len(icc_methods_psad)
icc_vals_psad = pd.DataFrame(np.ones((n, n)), index=icc_methods_psad, columns=icc_methods_psad)

for m1, m2 in combinations(icc_methods_psad, 2):
    long = pd.concat([
        psad[["ID", m1]].rename(columns={m1: "psad"}).assign(rater=m1),
        psad[["ID", m2]].rename(columns={m2: "psad"}).assign(rater=m2),
    ]).dropna(subset=["psad"])

    res = pg.intraclass_corr(data=long, targets="ID", raters="rater", ratings="psad", nan_policy='omit')
    row = res.set_index("Type").loc["ICC(A,1)"]
    icc = row["ICC"]

    icc_vals_psad.loc[m1, m2] = icc
    icc_vals_psad.loc[m2, m1] = icc

print("ICC(A,1) — two-way mixed, absolute agreement (PSAD, GT PSAD < 0.3)\n")
icc_vals_psad.style.background_gradient(cmap="RdYlGn", vmin=0.8, vmax=1).format(precision=3)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

cutoffs = [0.10, 0.15, 0.20]
bins = [0] + cutoffs + [np.inf]
bin_labels = ["<0.10", "0.10–0.15", "0.15–0.20", "0.20–0.30"]

gt_cat = pd.cut(psad["gt"], bins=bins, labels=bin_labels, right=False)

ncols = 4
nrows = -(-len(methods_with_manual) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(8, 3))
axes = axes.flatten()

for ax, method in zip(axes, methods_with_manual):
    pred_cat = pd.cut(psad[method], bins=bins, labels=bin_labels, right=False)

    valid = gt_cat.notna() & pred_cat.notna()
    cm = confusion_matrix(gt_cat[valid], pred_cat[valid], labels=bin_labels)

    kappa = cohen_kappa_score(gt_cat[valid], pred_cat[valid], weights="linear")
    kappa_quad = cohen_kappa_score(gt_cat[valid], pred_cat[valid], weights="quadratic")
    print(f"\n=== {method} ===")
    print(f"Cohen's kappa (linear):    {kappa:.3f}")
    # print(f"Cohen's kappa (quadratic): {kappa_quad:.3f}")
    print(classification_report(gt_cat[valid], pred_cat[valid], labels=bin_labels, zero_division=0))

    ax.imshow(cm, cmap="Blues")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(bin_labels, rotation=30, ha="right")
    ax.set_yticklabels(bin_labels)
    ax.set_xlabel("Predicted PSAD")
    ax.set_ylabel("GT PSAD")
    ax.set_title({"prostate_volume": "Manual"}.get(method, method))

for ax in axes[len(methods_with_manual):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

center = psad[(psad["gt"] >= 0.10) & (psad["gt"] < 0.20)]
gt_cat = center["gt"] > 0.15
for ax, method in zip(axes, methods_with_manual):
    pred_cat = center[method] >= 0.15
    cm = confusion_matrix(gt_cat, pred_cat)
    # print(cm)
    # print(
    #     (cm[0, 0] + cm[1, 1]) / cm.sum()
    # )

    kappa = cohen_kappa_score(gt_cat, pred_cat, weights="linear")
    print(f"\n=== {method} ===")
    print(f"Cohen's kappa (linear):    {kappa:.3f}")
    print(classification_report(gt_cat, pred_cat, zero_division=0))
